# Atlas recipe example

Shows a bundled Atlas plugin recipe on a tiny synthetic dataset.

Flow: create data -> load recipe -> check -> dry-run -> apply -> re-check.

In [ ]:
import woodpecker_atlas_plugin  # noqa: F401 - imports plugin fixes for editable installs

import woodpecker
from woodpecker.testing import make_atlas

Create an Atlas-like dataset with missing `project_id` metadata and overly strong compression.

In [ ]:
dataset = make_atlas(missing=["project_id"], seed=7)
dataset["pr"].encoding["complevel"] = 5

dataset

Load the bundled Atlas recipe and inspect its match rules and fix steps.

In [ ]:
recipe = woodpecker.recipe.get("c3s.atlas")

recipe.model_dump()

In [ ]:
recipe.match.model_dump(), [step.id for step in recipe.steps]

In [ ]:
findings = woodpecker.recipe.check(dataset, recipe)
findings.fix_ids

Dry-run previews the repair without changing the dataset.

In [ ]:
result = woodpecker.recipe.apply(dataset, recipe, dry_run=True)

result.stats, result.preview, dataset.attrs.get("project_id"), dataset["pr"].encoding["complevel"]

Apply the recipe in memory and re-check.

In [ ]:
write = woodpecker.recipe.apply(dataset, recipe, dry_run=False)

(
    write.stats,
    dataset.attrs["project_id"],
    dataset["pr"].encoding["complevel"],
    dataset["pr"].encoding["zlib"],
    dataset["pr"].encoding["shuffle"],
)

In [ ]:
recheck = woodpecker.recipe.check(dataset, recipe)
bool(recheck)